In [ ]:
# 用于检查 `reaction_path.csv` 中所有反应路径，并遵循显示规则：
# - 若"和反应物砌块分子反应的中间体分子"为 N/A/空/NaN，则仅显示"在zinc库里的反应物砌块分子"。
# - 否则显示"库存反应物 + 中间体反应物"。
#
# 使用滑动条来浏览所有路径。

In [2]:
import os, math
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, HTML
import ipywidgets as widgets

CSV_PATH = "../data/reaction_path.csv"
assert os.path.exists(CSV_PATH), f"CSV 文件未找到：{CSV_PATH}，请先生成。"

# 读取数据
df = pd.read_csv(CSV_PATH)
print(f"数据形状: {df.shape}")
print(f"列名: {list(df.columns)}")
print(f"共有 {len(df)} 条反应路径")

# 创建路径滑动条
path_slider = widgets.IntSlider(
    min=0, 
    max=len(df)-1, 
    step=1, 
    value=0, 
    description='路径索引:',
    style={'description_width': '100px'}
)

# 创建路径信息显示
path_info = widgets.HTML(
    value="<div>选择路径索引来查看反应详情</div>",
    layout=widgets.Layout(width='100%')
)

print("界面组件已创建")

def to_svg(smiles: str, w=200, h=200):
    """将SMILES转换为SVG图像"""
    try:
        mol = Chem.MolFromSmiles(smiles)
    except Exception:
        mol = None
    if mol is None:
        return "<div style='color:red; border:1px solid red; padding:10px;'>Invalid SMILES</div>"
    
    d = rdMolDraw2D.MolDraw2DSVG(w, h)
    rdMolDraw2D.PrepareAndDrawMolecule(d, mol)
    d.FinishDrawing()
    return d.GetDrawingText()

def _is_na_like(val):
    """检查值是否为NA-like"""
    if val is None:
        return True
    if isinstance(val, float) and math.isnan(val):
        return True
    if isinstance(val, str):
        s = val.strip()
        if not s:
            return True
        if s.lower() in ("n/a", "na", "nan"):
            return True
    return False

def parse_list(val: str):
    """解析反应物列表，用+分隔"""
    if _is_na_like(val):
        return []
    if not isinstance(val, str):
        return []
    return [x.strip() for x in val.split("+") if x.strip()]

print("辅助函数已定义")

def show_path(path_index):
    """显示指定索引的反应路径"""
    if path_index < 0 or path_index >= len(df):
        return "<div style='color:red;'>无效的路径索引</div>"
    
    row = df.iloc[path_index]
    
    # 获取反应信息
    product = str(row.get("当前状态分子", ""))
    in_stock_col = row.get("在zinc库里的反应物砌块分子", "")
    interm_col = row.get("和反应物砌块分子反应的中间体分子", "")
    template = str(row.get("反应模版", ""))
    
    # 显示规则：中间体 NA-like 时，仅显示库存反应物；否则显示库存 + 中间体
    if _is_na_like(interm_col):
        reactants = parse_list(in_stock_col)
    else:
        reactants = parse_list(in_stock_col) + parse_list(interm_col)
    
    # 生成SVG图像
    reactant_svgs = "".join([to_svg(smi) for smi in reactants])
    product_svg = to_svg(product)
    
    # 显示路径信息
    html = f"""
    <div style='margin:20px 0;'>
      <h3>路径 {path_index + 1} / {len(df)}</h3>
      <div style='margin-bottom:15px;'><b>反应模版</b>: {template}</div>
      
      <div style='display:flex; align-items:center; gap:20px; margin:20px 0;'>
        <div style='text-align:center;'>
          <div style='margin-bottom:10px; font-weight:bold;'>反应物</div>
          {reactant_svgs}
        </div>
        <div style='font-size: 40px; padding: 0 20px;'>→</div>
        <div style='text-align:center;'>
          <div style='margin-bottom:10px; font-weight:bold;'>产物</div>
          {product_svg}
        </div>
      </div>
      
      <div style='margin-top:20px;'>
        <div><b>库存反应物</b>: {in_stock_col}</div>
        <div><b>中间体反应物</b>: {interm_col}</div>
      </div>
    </div>
    """
    
    return html

print("显示函数已定义")

# 创建输出区域
output = widgets.Output()

def on_path_change(change):
    """当滑动条改变时更新显示"""
    with output:
        output.clear_output(wait=True)
        html_content = show_path(change['new'])
        display(HTML(html_content))

# 绑定事件
path_slider.observe(on_path_change, names='value')

print("事件已绑定")

# 显示界面
display(HTML("<h2>反应路径可视化</h2>"))
display(HTML("<p>使用滑动条浏览所有反应路径。每条路径显示反应物、产物和反应模版。</p>"))

display(path_slider)
display(path_info)
display(output)

# 初始显示
on_path_change({'new': 0})

print("界面已显示，可以开始使用了！")

数据形状: (4, 4)
列名: ['当前状态分子', '在zinc库里的反应物砌块分子', '和反应物砌块分子反应的中间体分子', '反应模版']
共有 4 条反应路径
界面组件已创建
辅助函数已定义
显示函数已定义
事件已绑定


IntSlider(value=0, description='路径索引:', max=3, style=SliderStyle(description_width='100px'))

HTML(value='<div>选择路径索引来查看反应详情</div>', layout=Layout(width='100%'))

Output()

界面已显示，可以开始使用了！
